# FABLE Pyculator 2021 Notebook Loop

This notebook uses the validated 2021 generated-model artifact. It intentionally uses a 2021 workbook path and a 2021 generated-model path. It does not fall back to the tracked 2020 generated model archive.

Expected local workbook artifact:

- `tmp/private-workbooks/2021_Open_FABLECalculator.xlsx`

Tracked compressed generated-model artifact:

- `examples/fable_2021/generated_fable_2021_model.py.xz`

Ignored generated-model working path:

- `tmp/generated-models/fable-2021/generated_fable_2021_model.py`


In [ ]:
from pathlib import Path
import lzma
import shutil

from fable_pyculator import (
    DEFAULT_2021_GENERATED_MODEL_PATH,
    DEFAULT_2021_WORKBOOK_PATH,
    build_2021_notebook_spec,
    output_table_frame,
    run_2021_notebook_loop,
)


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for path in (current, *current.parents):
        if (path / "pyproject.toml").exists() and (path / "src" / "fable_pyculator").exists():
            return path
    raise RuntimeError("Could not find fable-pyculator repository root")


repo_root = find_repo_root(Path.cwd())
workbook_path = repo_root / DEFAULT_2021_WORKBOOK_PATH
generated_model_path = repo_root / DEFAULT_2021_GENERATED_MODEL_PATH
archive_path = repo_root / "examples" / "fable_2021" / "generated_fable_2021_model.py.xz"

if not generated_model_path.exists() and archive_path.exists():
    generated_model_path.parent.mkdir(parents=True, exist_ok=True)
    with lzma.open(archive_path, "rb") as source, generated_model_path.open("wb") as target:
        shutil.copyfileobj(source, target)

missing = [path for path in (workbook_path, generated_model_path) if not path.exists()]
ARTIFACTS_READY = not missing

print(f"Workbook: {workbook_path}")
print(f"Generated model: {generated_model_path}")
print(f"Compressed generated model archive: {archive_path}")
if missing:
    print("Missing local artifact(s):")
    for path in missing:
        print(f"- {path}")
else:
    print("Ready: 2021 workbook and generated model are available.")


Build the 2021 wrapper spec. This discovers workbook structure only; it does not generate the Modelwright Python model.

In [ ]:
if ARTIFACTS_READY:
    spec = build_2021_notebook_spec(workbook_path)
    print(spec.workbook_id)
    print(len(spec.selection_controls), "selection controls")
    print(len(spec.scenario_definition_tables), "scenario definition tables")
    print(len(spec.output_tables), "output tables")
else:
    print("Restore the 2021 workbook and matching 2021 generated model before building the spec.")


Run the matching 2021 generated model and render notebook outputs.

In [ ]:
if ARTIFACTS_READY:
    result = run_2021_notebook_loop(
        {"gdp_scen": "SSP1"},
        workbook_path=workbook_path,
        generated_model_path=generated_model_path,
        output_table_column_flavour_tags="OUTPUT-*",
        include_figures=False,
    )
    print(result.run.name)
    print(len(result.output_tables), "rendered output tables")
    print(len(result.headline_frames), "headline frames")
else:
    print("Skipped model run because required local artifacts are missing.")


Inspect a focused GHG table after the run.

In [ ]:
if ARTIFACTS_READY:
    output_table_frame(result.run, "ghg_resultsghg", column_flavour_tags="OUTPUT-8").head()
else:
    print("Skipped output inspection because required local artifacts are missing.")
